In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from scipy.optimize import curve_fit
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
hppc_DIR = "../data/hppc/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_fit/"
res_DIR = "../data/results_fit/"
resistance_DIR = "../data/resistance/"
# %matplotlib widget

In [ ]:
t_in0 = 1
t_in1 = 1
t_inf = t_in0+t_in1
t_in = np.arange(0,t_inf,0.1)
# t_in = np.arange(0,t_inf,1)
# t_sim = np.arange(0,t_inf,0.01)
I_in = []
for tt in t_in:
    if tt<t_in0:
        I_in = np.append(I_in,0)
    elif tt>=t_in0 and tt<t_in0+t_in1:
        I_in = np.append(I_in,5)
# I_in = np.array([0,0,5,5,0,0,-5,-5,0,0])
parameter_values = get_parameter_values()
parameter_values = get_parameter_values()

spm = pybamm.lithium_ion.SPM(
    {
        "SEI": "ec reaction limited",
        "loss of active material": "stress-driven",
        "lithium plating": "irreversible",
        "stress-induced diffusion": "false",
        # "calculate discharge energy": "true"
    }
)
param=spm.param
timescale = parameter_values.evaluate(spm.timescale)

def get_pulse_res(parameter_values,esoh_sol,SOC):
    c_n_max = parameter_values.evaluate(param.n.prim.c_max)
    c_p_max = parameter_values.evaluate(param.p.prim.c_max)
    x_100 = esoh_sol["x_100"].data[0]
    y_100 = esoh_sol["y_100"].data[0]
    x_0 = esoh_sol["x_0"].data[0]
    y_0 = esoh_sol["y_0"].data[0]
    cs_n_0 = (SOC*(x_100-x_0)+x_0)*c_n_max
    cs_p_0 = (SOC*(y_100-y_0)+y_0)*c_p_max
    parameter_values.update(
      {
          "Initial concentration in negative electrode [mol.m-3]": cs_n_0,
          "Initial concentration in positive electrode [mol.m-3]": cs_p_0,        
      }
    )
    sim_pulse = pybamm.Simulation(spm, parameter_values=parameter_values, 
                            solver=pybamm.CasadiSolver(mode="safe", rtol=1e-6, atol=1e-6,dt_max=0.1))
    sol_pulse = sim_pulse.solve(t_eval=t_in)
    I   =  sol_pulse["Current [A]"].entries
    Vt  =  sol_pulse["Terminal voltage [V]"].entries
    idx = np.where(np.diff(np.sign(-I)))[0]
    Rs = abs((Vt[idx+1]-Vt[idx])/(I[idx+1]-I[idx]))[0]
    return Rs

def get_Rs(cyc_no,eSOH,parameter_values):
  model = spm
  Vmin = 3.0
  Vmax = 4.2
  esoh_model = pybamm.lithium_ion.ElectrodeSOH()
  esoh_sim = pybamm.Simulation(esoh_model, parameter_values=parameter_values)
  Cn = eSOH["C_n"][Ns[cyc_no]]
  Cp = eSOH["C_p"][Ns[cyc_no]]
  c_n_max = parameter_values.evaluate(param.n.prim.c_max)
  c_p_max = parameter_values.evaluate(param.p.prim.c_max)
  n_Li_init = eSOH["Total lithium in particles [mol]"][Ns[cyc_no]]
  c_plated_Li = eSOH['X-averaged lithium plating concentration [mol.m-3]'][Ns[cyc_no]]
  eps_n_data = parameter_values.evaluate(Cn*3600/(param.n.L * param.n.prim.c_max * param.F* param.A_cc))
  eps_p_data = parameter_values.evaluate(Cp*3600/(param.p.L * param.p.prim.c_max * param.F* param.A_cc))
  del_sei = eSOH['X-averaged SEI thickness [m]'][Ns[cyc_no]]
  esoh_sol = esoh_sim.solve(
      [0],
      inputs={"V_min": Vmin, "V_max": Vmax, "C_n": Cn, "C_p": Cp, "n_Li": n_Li_init},
      solver=pybamm.AlgebraicSolver(),
  )
  parameter_values.update(
          {
              "Negative electrode active material volume fraction": eps_n_data,
              "Positive electrode active material volume fraction": eps_p_data,
              "Initial inner SEI thickness [m]": 0e-09,
              "Initial outer SEI thickness [m]": del_sei,
              "Initial plated lithium concentration [mol.m-3]": c_plated_Li,
          }
        )
  timescale = parameter_values.evaluate(spm.timescale)
  current_interpolant = pybamm.Interpolant(
    t_in, -I_in, timescale * pybamm.t
  )
  parameter_values["Current function [A]"] = current_interpolant
  SOC_vals = np.linspace(1,0,11)
  Rs_ch_s = []
  for SOC in SOC_vals[1:10]:
      Rs_t = get_pulse_res(parameter_values,esoh_sol,SOC)
      Rs_ch_s.append(Rs_t)
  Rs_ch = np.average(Rs_ch_s)
  
  current_interpolant = pybamm.Interpolant(
    t_in, I_in, timescale * pybamm.t
  )
  parameter_values["Current function [A]"] = current_interpolant
  SOC_vals = np.linspace(1,0,11)
  Rs_dh_s = []
  for SOC in SOC_vals[1:10]:
      Rs_t = get_pulse_res(parameter_values,esoh_sol,SOC)
      Rs_dh_s.append(Rs_t)
  Rs_dh = np.average(Rs_dh_s)
  Rs = (Rs_dh + Rs_ch)/2
  return Rs

In [ ]:
cells = [3,9,12]
cells = [3,6,9,12,15,18]
sno = 11
sim_des = 'low_init_SEI'
sim_des = f'cond{sno}'
o_SEI = 1.25*3e4
o_lip = 5*3e4
# sim_des = sim_des+'_cv'
for cell in cells:
    # sim_des = sim_des+'_cv'
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    print(cell_no)
    Ns = np.insert(N_0[1:]-1,0,0)
    eps_n_data,eps_p_data,c_rate_c,c_rate_d,dis_set,Temp,SOC_0 = init_exp(cell_no,dfe_0,spm,parameter_values)
    pybamm.set_logging_level("WARNING")
    # pybamm.set_logging_level("NOTICE")
    experiment = pybamm.Experiment(
        [
            ("Discharge at "+c_rate_d+dis_set,
            "Rest for 10 sec",
            "Charge at "+c_rate_c+" until 4.2V", 
            "Hold at 4.2V until C/100")
        ] *(dfe_0.N.iloc[-1]),
        termination="50% capacity",
    #     cccv_handling="ode",
    )
    par_val = {}
    # Room temp
    par_val[0] = [4.0312e-08,1.8157e-07,1.0776,2.3586e-09,-4.9170e-09,-1.4406e-09,4.60788219e-16,4.56607447e-19]
    # First Tuning
    par_val[1] = [5.6076e-08,6.7429e-07,1.02,1.4576e-08,-1.7447e-07,-2.5257e-08,4.60788219e-16,4.56607447e-19]
    # Contraint 1/2 and 2
    par_val[2] = [8.0624e-08,3.6314e-07,1.02,4.7172e-09,-9.8340e-09,-2.8812e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5
    par_val[3] = [9.8220e-08,9.0785e-07,1.02,1.1793e-08,-2.4585e-08,-7.2030e-09,4.60788219e-16,4.56607447e-19]
    # Constraint 1/5 and 5 with no constraints on beta2'
    par_val[4] = [6.2621e-08,6.8771e-07,1.02,1.1793e-08,-1.7385e-07,-2.4901e-08,4.60788219e-16,4.56607447e-19]
    # Constraint 1/2 and 2 with no constraints on beta2'
    par_val[5] = [8.0624e-08,3.6314e-07,1.002,4.7172e-09,-2.4217e-07,-2.5212e-08,4.60788219e-16,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[6] = [1.2796e-07,7.2624e-07,1.02,0,-1.7764e-07,-1.4406e-11,4.9166e-17,4.56607447e-19]
    # kpl=0, retune ksei
    par_val[7] = [1.1375e-07,6.4051e-07,1.0,0,-1.5557e-07,-2.2287e-08,2.2675e-15,4.56607447e-19]
    # kpl=0, retune ksei, different cost function, better initial guess    
    par_val[8] = [9.1084e-08,5.9270e-07,1.00,0,-1.6967e-07,-3.0447e-08,1.5803e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 6 9 12
    par_val[9] = [2.8391e-07,2.0958e-06,1.9123,0,-8.6507e-07,-9.9184e-08,2.8281e-15,4.56607447e-19]
    # kpl=0, retune ksei, cells 3 9 12
    par_val[10] = [1.3292e-07,9.2142e-07,1.0,0,-1.2058e-07,-2.1779e-08,1.9308e-15,4.56607447e-19]
    # 2 step method with ksei
    par_val[11] = [1.1759e-07,8.9155e-07,1.0,7.5992e-09,-1.2611e-07,-2.3971e-08,1.4840e-15,4.56607447e-19]
    # 2 step method with dsei
    par_val[12] = [1.2057e-07,8.9269e-07,1.0,1.1355e-08,-1.3511e-07,-3.1561e-08,4.6079e-16,3.6445e-18]
    # 1 step method with dsei
    par_val[13] = [1.4653e-07,9.3369e-07,1.0,4.2122e-09,-1.3385e-07,-2.7939e-08,4.6079e-16,4.7510e-19]
    # First Tuning 3,9,12
    par_val[14] = [1.1944e-07,8.9283e-07,1.0,1.1646e-08,-1.3499e-07,-3.0872e-08,4.60788219e-16,4.56607447e-19]
    parameter_values = get_parameter_values()
    parameter_values.update(
        {
            "Negative electrode active material volume fraction": eps_n_data,
            "Positive electrode active material volume fraction": eps_p_data,
            "Initial temperature [K]": 273.15+Temp,
            "Ambient temperature [K]": 273.15+Temp,
            "Positive electrode LAM constant proportional term [s-1]": par_val[sno][0],
            "Negative electrode LAM constant proportional term [s-1]": par_val[sno][1],
            "Positive electrode LAM constant proportional term 2 [s-1]": par_val[sno][5],
            "Negative electrode LAM constant proportional term 2 [s-1]": par_val[sno][4],
            "Positive electrode LAM constant exponential term": par_val[sno][2],
            "Negative electrode LAM constant exponential term": par_val[sno][2],
            "SEI kinetic rate constant [m.s-1]":  par_val[sno][6], #1.08494281e-16 , 
            "EC diffusivity [m2.s-1]": par_val[sno][7],#8.30909086e-19,
            "SEI growth activation energy [J.mol-1]": 1.87422275e+04,#1.58777981e+04,
            "Lithium plating kinetic rate constant [m.s-1]": par_val[sno][3],
            "Initial inner SEI thickness [m]": 0e-09,
            "Initial outer SEI thickness [m]": 5e-09,
            "Li plating resistivity [Ohm.m]": o_lip,
            "SEI resistivity [Ohm.m]": o_SEI,
            "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
            "Negative electrode LAM min stress [Pa]": 0,
            "Negative electrode LAM max stress [Pa]": 0,
            "Positive electrode LAM min stress [Pa]": 0,
            "Positive electrode LAM max stress [Pa]": 0,
            "Negative electrode diffusion coefficient [m2.s-1]": 8e-14,
            "Positive electrode diffusion coefficient [m2.s-1]": 8e-15,
            # "Negative electrode critical stress [Pa]": 20e+06,
            # "Positive electrode critical stress [Pa]": 40e+06,
        },
        check_already_exists=False,
    )
    all_sumvars_dict = cycle_adaptive_simulation_V2(spm, parameter_values, experiment,SOC_0, save_at_cycles=1)
    # plotc(all_sumvars_dict,dfe);
    i = 0
    Rs = []
    # for cyc_no in [0,int((len(N)+1)/2),len(N)-2]:
    # for cyc_no in [0,len(N)-2]:
    for cyc_no in range(len(Ns)):
        Rs_t = get_Rs(cyc_no,all_sumvars_dict,parameter_values)
        Rs.append(Rs_t)

    df = pd.DataFrame({'N': N_0,'Ah_th':dfe_0["Ah_th"]-dfe_0["Ah_th"][0], 'Rs_data': dfe_0["Rs_ave"],'Rs_sim':Rs,
                })
    df.to_csv(res_DIR + "DC_resistance"+sim_des+"_cell_"+cell_no+".csv", index=False)
    

In [ ]:
for cell in cells:
    # sim_des = sim_des+'_cv'
    cell_no,dfe,dfe_0,dfo_0,N,N_0 = load_data(cell,eSOH_DIR,oCV_DIR)
    df = pd.read_csv(res_DIR + "DC_resistance"+sim_des+"_cell_"+cell_no+".csv")
    fig, ax = plt.subplots(1,1,figsize=(5,4))
    ax.plot(df["N"],(df["Rs_data"]-df["Rs_data"].iloc[0]),'kx-')
    ax.plot(df["N"],(df["Rs_sim"]-df["Rs_sim"].iloc[0]),'rx-')
    # ax.plot(df["N"],(df["Rs_data"]),'kx-')
    # ax.plot(df["N"],(df["Rs_sim"]),'rx-')
    ax.set_xlabel('Cycle Number')
    # ax.set_ylabel(r'DCR Average [$\Omega$]')
    ax.set_ylabel(r'Resistance Growth $\Delta R$ [$\Omega$]')
    ax.legend(['Data','Sim'])
    # ax.set_title("Cycling Aging Resistance")
    # plt.savefig(fig_DIR +'cycling_aging_cell_'+cell_no+'_resistance_1.png')
    plt.savefig(fig_DIR +'cycling_aging'+sim_des+'_cell_'+cell_no+'_resistance.png')

In [ ]:
asdas

In [ ]:
def get_Rs_SOC(cyc_no,eSOH,parameter_values):
  model = spm
  Vmin = 3.0
  Vmax = 4.2
  esoh_model = pybamm.lithium_ion.ElectrodeSOH()
  esoh_sim = pybamm.Simulation(esoh_model, parameter_values=parameter_values)
  Cn = eSOH["C_n"][Ns[cyc_no]]
      # print(Cn)
  Cp = eSOH["C_p"][Ns[cyc_no]]
  c_n_max = parameter_values.evaluate(param.n.prim.c_max)
  c_p_max = parameter_values.evaluate(param.p.prim.c_max)
  n_Li_init = eSOH["Total lithium in particles [mol]"][Ns[cyc_no]]
  eps_n_data = parameter_values.evaluate(Cn*3600/(param.n.L * param.n.prim.c_max * param.F* param.A_cc))
  eps_p_data = parameter_values.evaluate(Cp*3600/(param.p.L * param.p.prim.c_max * param.F* param.A_cc))
  del_sei = eSOH['X-averaged SEI thickness [m]'][Ns[cyc_no]]
  c_plated_Li = eSOH['X-averaged lithium plating concentration [mol.m-3]'][Ns[cyc_no]]
  esoh_sol = esoh_sim.solve(
      [0],
      inputs={"V_min": Vmin, "V_max": Vmax, "C_n": Cn, "C_p": Cp, "n_Li": n_Li_init},
      solver=pybamm.AlgebraicSolver(),
  )
  parameter_values.update(
          {
              "Negative electrode active material volume fraction": eps_n_data,
              "Positive electrode active material volume fraction": eps_p_data,
              "Initial temperature [K]": 273.15+25,
              "Ambient temperature [K]": 273.15+25,
              "Initial inner SEI thickness [m]": 0e-09,
              "Initial outer SEI thickness [m]": del_sei,
              "Initial plated lithium concentration [mol.m-3]": c_plated_Li,
          }
        )
  timescale = parameter_values.evaluate(spm.timescale)
  current_interpolant = pybamm.Interpolant(
    t_in, -I_in, timescale * pybamm.t
  )
  parameter_values["Current function [A]"] = current_interpolant
  SOC_vals = np.linspace(1,0,11)
  Rs_ch_s = []
  for SOC in SOC_vals[1:10]:
      Rs_t = get_pulse_res(parameter_values,esoh_sol,SOC)
      Rs_ch_s.append(Rs_t)
  Rs_ch = np.average(Rs_ch_s)
  
  current_interpolant = pybamm.Interpolant(
    t_in, I_in, timescale * pybamm.t
  )
  parameter_values["Current function [A]"] = current_interpolant
  SOC_vals = np.linspace(1,0,11)
  Rs_dh_s = []
  for SOC in SOC_vals[1:10]:
      Rs_t = get_pulse_res(parameter_values,esoh_sol,SOC)
      Rs_dh_s.append(Rs_t)
  Rs_dh = np.average(Rs_dh_s)
  Rs = (Rs_dh + Rs_ch)/2
  Rs_ave_s = (np.array(Rs_dh_s)+np.array(Rs_ch_s))/2
  return Rs_ave_s,Rs_ch_s,Rs_dh_s

In [ ]:
soc = np.linspace(0.9,0.1,9)

In [ ]:
i=0
titles = ["BOL","MOL","EOL"]
fig, ax = plt.subplots(1,3,figsize=(10,3))
for cyc_no in [0,int((len(N)+1)/2),len(N)-2]:
# for cyc_no in [0]:
    parameter_values.update(
    {
        "Positive electrode LAM constant proportional term [s-1]": par_val[1][0],
        "Negative electrode LAM constant proportional term [s-1]": par_val[1][1],
        "Positive electrode LAM constant exponential term": par_val[1][2],
        "Negative electrode LAM constant exponential term": par_val[1][2],
        "Lithium plating kinetic rate constant [m.s-1]": par_val[1][3],
    },
    check_already_exists=False,
    )
    Rs_ave,Rs_ch,Rs_dh = get_Rs_SOC(cyc_no,all_sumvars_dict,parameter_values)
    res_data = pd.read_csv(resistance_DIR+'resistance_data_cell_'+cell_no+'.csv', header=None).to_numpy()
    res_data[res_data == 0] = 'nan'
    ax1 = ax.flat[i]
    ax1.plot(soc,res_data[cyc_no],'kx-')
    ax1.plot(soc,Rs_ave,'rx--')
    ax1.set_xlabel('SOC')
    ax1.set_ylabel(r'DCR Average [$\Omega$]')
    ax1.set_title('{} [N:{:0.0f},Ah:{:0.0f},%Cap:{:0.1f}]'.format(titles[i],N[cyc_no],dfe['Ah_th'][cyc_no],(dfe["Cap"][cyc_no]/dfe["Cap"][0])*100))
    i+=1
ax1.legend(["Data","Sim"])
fig.suptitle('DCR Ave vs SOC')
fig.tight_layout()
# ax.legend(['Data','Sim'])
# plt.savefig(fig_DIR +'cycling_aging_cell_'+cell_no+'_resistance_1.png')
plt.savefig(fig_DIR +'resistance_SOC_cell_'+cell_no+'.png')

In [ ]:
i=0
titles = ["BOL","MOL","EOL"]
fig, ax = plt.subplots(1,1,figsize=(5,4))
for cyc_no in [0,int((len(N)+1)/2),len(N)-2]:
# for cyc_no in [0]:
    parameter_values.update(
    {
        "Positive electrode LAM constant proportional term [s-1]": par_val[1][0],
        "Negative electrode LAM constant proportional term [s-1]": par_val[1][1],
        "Positive electrode LAM constant exponential term": par_val[1][2],
        "Negative electrode LAM constant exponential term": par_val[1][2],
        "Lithium plating kinetic rate constant [m.s-1]": par_val[1][3],
    },
    check_already_exists=False,
    )
    Rs_ave,Rs_ch,Rs_dh = get_Rs_SOC(cyc_no,all_sumvars_dict,parameter_values)
    res_data = pd.read_csv(resistance_DIR+'resistance_data_cell_'+cell_no+'.csv', header=None).to_numpy()
    res_data[res_data == 0] = 'nan'
    ax1 = ax
    ax1.plot(soc,res_data[cyc_no],'kx-')
    ax1.plot(soc,Rs_ave,'rx--')
    ax1.set_xlabel('SOC')
    ax1.set_ylabel(r'DCR Average [$\Omega$]')
    # ax1.set_title('{} [N:{:0.0f},Ah:{:0.0f},%Cap:{:0.1f}]'.format(titles[i],N[cyc_no],dfe['Ah_th'][cyc_no],(dfe["Cap"][cyc_no]/dfe["Cap"][0])*100))
    i+=1
ax1.legend(["Data","Sim"])
# fig.suptitle('DCR Ave vs SOC')
fig.tight_layout()
# ax.legend(['Data','Sim'])
# plt.savefig(fig_DIR +'cycling_aging_cell_'+cell_no+'_resistance_1.png')
plt.savefig(fig_DIR +'resistance_SOC_cell_'+cell_no+'_all.png')

# Plots all Cells

In [ ]:
dfgdfgfd

In [ ]:
cells = [1,4,7,10,13,16,19]
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
for cell in cells:
    cell_no = f'{cell:02d}'
    df = pd.read_csv(res_DIR + "DC_resistance_cell_"+cell_no+".csv")
    ax.plot(df["Ah_th"],df["Rs_data"],'k',linestyle="None",marker=markers[i])
    ax.plot(df["Ah_th"],df["Rs_sim"],'r',linestyle="None",marker=markers[i])
    i+=1
ax.set_xlabel('Cycle Number')
ax.set_ylabel(r'Resistance [$\Omega$]')
ax.set_title(r'Average DC Resistance')
ax.legend(['Data','Sim'])
# plt.savefig(fig_DIR +'cycling_aging_room_resistance.png')

In [ ]:
cells = [1,4,7,10,13,16,19]
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
colors = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown","tab:cyan"]
for cell in cells:
    cell_no = f'{cell:02d}'
    df = pd.read_csv(res_DIR + "DC_resistance_cell_"+cell_no+".csv")
    ax.plot(df["Ah_th"],df["Rs_data"],linestyle="None",marker=markers[i],label='_nolegend_',color=colors[i])
    ax.plot(df["Ah_th"],df["Rs_sim"],color=colors[i])
    i+=1
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r'Resistance [$\Omega$]')
ax.set_title(r'Average DC Resistance')
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.13], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'cycling_aging_room_resistance.png')

In [ ]:
cells = [1,4,7,10]
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
colors = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown","tab:cyan"]
for cell in cells:
    cell_no = f'{cell:02d}'
    df = pd.read_csv(res_DIR + "DC_resistance_cell_"+cell_no+".csv")
    ax.plot(df["Ah_th"],df["Rs_data"],linestyle="None",marker="o",label='_nolegend_',color=colors[i])
    ax.plot(df["Ah_th"],df["Rs_sim"],color=colors[i])
    i+=1
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r'Resistance [$\Omega$]')
ax.set_title(r'Average DC Resistance')
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'cycling_aging_room_resistance_s1_1.png')

In [ ]:
cells = [1,13,10,16]
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
colors = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown","tab:cyan"]
for cell in cells:
    cell_no = f'{cell:02d}'
    df = pd.read_csv(res_DIR + "DC_resistance_cell_"+cell_no+".csv")
    ax.plot(df["Ah_th"],df["Rs_data"],linestyle="None",marker="o",label='_nolegend_',color=colors[i])
    ax.plot(df["Ah_th"],df["Rs_sim"],color=colors[i])
    i+=1
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r'Resistance [$\Omega$]')
ax.set_title(r'Average DC Resistance')
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['C/5','C/5 50% DOD','Mixed Crate','Mixed 50% DOD'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'cycling_aging_room_resistance_s1_2.png')

In [ ]:
cells = [13,16,19]
fig, ax = plt.subplots(1,1,figsize=(6,4))
i = 0
markers = ["o","v","^","1","*","d","p"]
colors = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown","tab:cyan"]
for cell in cells:
    cell_no = f'{cell:02d}'
    df = pd.read_csv(res_DIR + "DC_resistance_cell_"+cell_no+".csv")
    ax.plot(df["Ah_th"],df["Rs_data"],linestyle="None",marker="o",label='_nolegend_',color=colors[i])
    ax.plot(df["Ah_th"],df["Rs_sim"],color=colors[i])
    i+=1
ax.set_xlabel('Ah Throughput')
ax.set_ylabel(r'Resistance [$\Omega$]')
ax.set_title(r'Average DC Resistance')
# ax.legend(['C/5','1.5C','2C','Mixed Crate','C/5 50% DOD','Mixed 50% DOD','Drive Cycle'])
fig.legend(['C/5 50% DOD','Mixed 50% DOD','Drive Cycle'], 
            loc="lower center",bbox_to_anchor=[0.5,-0.10], ncol=4, fontsize=11)
plt.savefig(fig_DIR +'cycling_aging_room_resistance_s2_2.png')